[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler/blob/main/prep/P4_package_release.ipynb)


# P4 — จัดแพ็กข้อมูล: repo (วันหลัก + ตารางทั้งเดือน) และ Zenodo (กริดรายวันทั้งเดือน)

**สำหรับผู้ดูแลข้อมูล** — ขั้นสุดท้าย

1. ตรวจไฟล์รายวันทั้งหมด (CAPPI, polar, labels) และคำนวณ SHA-256
2. เลือก **วันหลัก** ที่จะเก็บกริดไว้ใน repo (ไม่เกิน `repo_max_mb`) — นิสิตรันบท 02–06 ได้ทันทีโดยไม่ต้องดาวน์โหลด
3. เขียน `data/derived/day_index.csv` และ `config/data_sources.yaml` ให้ `utils/data.py` รู้ว่าไฟล์ไหนอยู่ที่ไหน
4. (เลือกได้) อัปโหลดขึ้น Zenodo แบบ **draft** ด้วย token ใน Colab Secrets (`ZENODO_TOKEN`) แล้วไปตรวจและกด Publish เองบนเว็บ

> ข้อมูลที่เผยแพร่เป็นกริด Cartesian 1 km ที่ประมวลผลแล้วเท่านั้น ไม่มีไฟล์ UF หรือข้อมูลในพิกัด polar

In [ ]:
# @title ⚙️ เตรียมสภาพแวดล้อม (รันก่อนเสมอ)
import os, sys, subprocess
REPO = "Teach_stormTracks_KKN_May2024_HID_doppler"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/nattaponm/{REPO}.git", f"/content/{REPO}"], check=True)
    else:
        subprocess.run(["git", "-C", f"/content/{REPO}", "pull", "-q"])
    os.chdir(f"/content/{REPO}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "prep"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("โฟลเดอร์ทำงาน:", os.getcwd(), "| CPU", os.cpu_count(), "core")

In [ ]:
import json, time, shutil, warnings
from pathlib import Path
from functools import partial
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

from utils.config import load_settings, repo_path
from utils import maps as M
s = load_settings()
M.setup_style()
OUT_ROOT = Path(os.environ.get("KKN_OUT_ROOT", s["radar_data"]["drive_derived_root"]))
N_WORKERS = os.cpu_count()
CTX = mproc.get_context("fork")

# ดึงผลของขั้นก่อนหน้าจาก Drive (Colab แยกเครื่องต่อ notebook — ดู utils/sync.py)
from utils import sync
sync.pull(OUT_ROOT)
s = load_settings()      # อ่านใหม่หลัง pull (พิกัดเรดาร์จริงจาก P0)

import hashlib, yaml
from utils import data_io
DER, DAYS_DIR = repo_path("data", "derived"), repo_path("data", "days")
KINDS = {"cappi_15min": OUT_ROOT / "cappi", "polar_15min": OUT_ROOT / "polar"}
KINDS.update({f"labels_{t}": OUT_ROOT / "tracks" for t in s["tracking"]["tags"]})

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

rows = []
for kind, folder in KINDS.items():
    for p in sorted(folder.glob(f"KKN_*_{kind}.nc")):
        rows.append({"day": pd.Timestamp(p.name[4:12]), "kind": kind, "file": p.name, "path": str(p),
                     "mb": round(p.stat().st_size / 1e6, 3), "sha256": sha256(p)})
files = pd.DataFrame(rows)
print(f"{len(files)} ไฟล์ รวม {files['mb'].sum():.0f} MB")
files.pivot_table(index="kind", values="mb", aggfunc=["count", "sum"]).round(1)

## 1. เลือกวันหลักสำหรับ repo

ค่าตั้งต้น: 3 วันที่มีคะแนนความเสียหายในรัศมีสูงสุด (จากบท 01) + 1 วันในช่วงต้นฤดูฝนที่มีเส้นทาง ≥ 30 นาทีมากที่สุด (เป็นกลุ่มเปรียบเทียบ) ตัดออกทีละวันถ้าเกิน `repo_max_mb`

In [ ]:
key = s["release"]["key_days"]
if key is None:
    rp = repo_path("data", "reports", "days_may2024.csv")
    if not rp.exists():
        raise FileNotFoundError("ไม่พบ data/reports/days_may2024.csv — รันบท 01 โดยตั้ง ADMIN_SYNC = True (เก็บผลเข้า Drive) "
                                "หรือ commit data/reports ก่อน หรือกำหนด release.key_days ใน config/settings.yaml เอง")
    rep_days = pd.read_csv(rp, parse_dates=["date"])
    key = list(rep_days.sort_values("D_in_range", ascending=False)["date"].head(3))
    T15 = pd.read_csv(DER / "tracks_15min.csv", parse_dates=["day"])
    rainy = T15[(T15["day"].dt.day >= 21) & (T15["lifetime_min"] >= 30)].groupby("day").size()
    if len(rainy):
        key.append(rainy.idxmax())
key = [pd.Timestamp(k) for k in key if pd.Timestamp(k) in set(files["day"])]
size = files[files["day"].isin(key)].groupby("day")["mb"].sum()
while size.sum() > s["release"]["repo_max_mb"] and len(size) > 1:
    size = size.iloc[:-1]
key = list(size.index)
print("วันหลัก:", [f"{k:%d %b}" for k in key], f"| {size.sum():.1f} MB")
for _, r in files[files["day"].isin(key)].iterrows():
    dst = DAYS_DIR / f"{r['day']:%Y%m%d}" / r["file"]
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(r["path"], dst)

## 2. ดัชนีวันและแหล่งข้อมูล

In [ ]:
idx = files.groupby("day").agg(kinds=("kind", lambda k: ";".join(sorted(k))), mb=("mb", "sum")).reset_index()
idx["in_repo"] = idx["day"].isin(key)
idx["in_zenodo"] = True
idx.to_csv(DER / "day_index.csv", index=False)
src_path = repo_path("config", "data_sources.yaml")
old = yaml.safe_load(open(src_path, encoding="utf-8")) if src_path.exists() else {}
zen = (old or {}).get("zenodo") or {}
zen.update({"record_id": zen.get("record_id"), "doi": zen.get("doi"), "sandbox": s["release"]["zenodo_sandbox"],
            "files": {r["file"]: {"mb": float(r["mb"]), "sha256": r["sha256"]} for _, r in files.iterrows()}})
with open(src_path, "w", encoding="utf-8") as f:
    yaml.safe_dump({"zenodo": zen}, f, allow_unicode=True, sort_keys=False)
manifest = {"created_utc": pd.Timestamp.utcnow().isoformat(), "n_files": len(files), "total_mb": float(files["mb"].sum()),
            "key_days_in_repo": [f"{k:%Y-%m-%d}" for k in key],
            "derived_tables": sorted(p.name for p in DER.glob("*.*")),
            "processing": "prep/P0–P4 (Teach_stormTracks_KKN_May2024_HID_doppler)", "license": "CC BY 4.0",
            "radar_data_credit": "Thai Meteorological Department (KKN); UF files not distributed"}
json.dump(manifest, open(DER / "manifest.json", "w"), indent=1, ensure_ascii=False)
print("เขียน day_index.csv, config/data_sources.yaml, manifest.json")

## 3. อัปโหลด Zenodo (draft)

1. สร้าง token ที่ Zenodo → Applications → Personal access tokens (สิทธิ์ `deposit:write`, `deposit:actions`)
2. ใน Colab: แถบซ้าย 🔑 Secrets → เพิ่ม `ZENODO_TOKEN`
3. ตั้ง `UPLOAD = True` แล้วรัน ไฟล์จะขึ้นเป็น **draft** ยังไม่เผยแพร่ ตรวจ metadata บนเว็บแล้วกด Publish เอง
4. คัดลอก record id และ DOI มาใส่ใน `config/data_sources.yaml` แล้ว commit (นิสิตจะดาวน์โหลดวันอื่นได้อัตโนมัติ)

In [ ]:
UPLOAD = False
if UPLOAD:
    import requests
    from google.colab import userdata
    token = userdata.get("ZENODO_TOKEN")
    base = "https://sandbox.zenodo.org" if s["release"]["zenodo_sandbox"] else "https://zenodo.org"
    hdr = {"Authorization": f"Bearer {token}"}
    dep = requests.post(f"{base}/api/deposit/depositions", json={}, headers=hdr).json()
    bucket = dep["links"]["bucket"]
    for _, r in files.iterrows():
        with open(r["path"], "rb") as fp:
            resp = requests.put(f"{bucket}/{r['file']}", data=fp, headers=hdr)
        print(r["file"], resp.status_code)
    meta = {"metadata": {
        "title": "KKN weather radar gridded products, May 2024: pseudo-CAPPI, dual-pol/Doppler/HID and T-DaTing storm tracks",
        "upload_type": "dataset", "license": "cc-by-4.0",
        "description": ("Daily 1-km gridded products from the Khon Kaen (KKN) C-band dual-polarization radar of the Thai "
                        "Meteorological Department for May 2024 (pre-monsoon to rainy-season transition). Raw UF volumes are "
                        "not included. Processing code: https://github.com/nattaponm/Teach_stormTracks_KKN_May2024_HID_doppler"),
        "creators": [{"name": "แก้ชื่อผู้สร้าง, ชื่อ"}],
        "keywords": ["weather radar", "dual polarization", "hydrometeor classification", "storm tracking", "Thailand"]}}
    requests.put(f"{base}/api/deposit/depositions/{dep['id']}", json=meta, headers=hdr)
    print("draft:", dep["links"]["html"], "| record id:", dep["id"])

In [ ]:
# สรุปสิ่งที่ต้อง commit เข้า repo
tot = 0
for p in sorted(repo_path("data").rglob("*")):
    if p.is_file() and "cache" not in p.parts:
        tot += p.stat().st_size
print(f"ขนาด data/ ใน repo ทั้งหมด {tot / 1e6:.1f} MB")
for d in ("derived", "days", "sounding", "inventory"):
    q = repo_path("data", d)
    if q.exists():
        print(f"  data/{d}: {sum(f.stat().st_size for f in q.rglob('*') if f.is_file()) / 1e6:.1f} MB")

## บันทึกผลเข้า Drive และ (เลือกได้) commit ขึ้น GitHub

`sync.push` เก็บไฟล์ที่จะเข้า repo ไว้ใน `KKN_derived_v1/repo_mirror/` notebook ถัดไป (เครื่องใหม่) จะดึงไปใช้เองตอนเริ่ม **รันเซลล์นี้ทุกครั้งก่อนปิด runtime**

ถ้าต้องการ commit ทันที ตั้ง `COMMIT = True` (ต้องมี secret `KKN_commit` = fine-grained token ของ repo นี้ สิทธิ์ Contents: Read and write) ตั้ง `dry_run=True` เพื่อดูรายการไฟล์ก่อน — ฟังก์ชันไม่พิมพ์ token ออกมา

In [ ]:
sync.push(OUT_ROOT)
COMMIT = False
if COMMIT:
    sync.git_push("P4: key days, day index, data sources", dry_run=False)